# 02. 수신 통화 분류 결과 시각화

`01_analyze_inbound_calls.ipynb`의 분류 결과(`data/inbound/inbound_call_classification.json`)를 읽어
11개의 분석 이미지를 생성합니다. 챗봇 시나리오 설계를 위한 인사이트 도출용입니다.

- **입력**: `data/inbound/inbound_call_classification.json`
- **출력**: `output/inbound/01_overview.png` ~ `11_time_of_day.png` (11개)


In [ ]:
import os
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings("ignore")

sns.set_style("whitegrid", {"font.family": ["Malgun Gothic"]})
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.facecolor"] = "white"

LABEL_COLORS = {"상담": "#2196F3", "비상담": "#FF9800", "불명확": "#9E9E9E", "광고": "#F44336"}
CAT_COLORS = [
    "#E91E63", "#2196F3", "#4CAF50", "#FF9800", "#9C27B0",
    "#00BCD4", "#FF5722", "#795548", "#607D8B", "#9E9E9E",
]

DATA_PATH = "data/inbound/inbound_call_classification.json"
OUTPUT_DIR = "output/inbound"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("환경 설정 완료")


In [ ]:
# ============================================================
# 데이터 로드
# ============================================================
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

df = pd.DataFrame(raw)
df["folder"] = df["folder"].astype(str)
df["month"] = df["folder"].str[:6]
df["year_month"] = pd.to_datetime(df["month"], format="%Y%m")
df["qa_count"] = df["qa_pairs"].apply(lambda x: len(x) if isinstance(x, list) else 0)
df["duration_min"] = df["duration_sec"] / 60

consult_df = df[df["label"] == "상담"].copy()

# QA 전체 펼치기
all_qa = []
for _, row in consult_df.iterrows():
    for qa in (row["qa_pairs"] or []):
        if isinstance(qa, dict) and qa.get("question") and qa.get("answer"):
            all_qa.append({
                "month": row["month"],
                "year_month": row["year_month"],
                "duration_sec": row["duration_sec"],
                "question": qa.get("question", ""),
                "answer": qa.get("answer", ""),
                "q_len": len(qa.get("question", "")),
                "a_len": len(qa.get("answer", "")),
            })
qa_df = pd.DataFrame(all_qa)

print("=" * 50)
print("데이터 로드 완료")
print("=" * 50)
print(f"전체 통화:    {len(df):>5,}건")
print(f"상담:         {(df.label=='상담').sum():>5,}건 ({(df.label=='상담').sum()/len(df)*100:.1f}%)")
print(f"비상담:       {(df.label=='비상담').sum():>5,}건")
print(f"불명확:       {(df.label=='불명확').sum():>5,}건")
print(f"광고:         {(df.label=='광고').sum():>5,}건")
print(f"분석 기간:    {df.month.min()} ~ {df.month.max()}")
print(f"총 QA 쌍:     {len(qa_df):>5,}개")
print(f"상담당 평균QA: {consult_df.qa_count.mean():.2f}개")


## 1. 전체 현황 개요

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("기프트코 통화 데이터 전체 현황", fontsize=16, fontweight="bold")

ax1 = axes[0, 0]
lc = df["label"].value_counts()
colors_pie = [LABEL_COLORS.get(l, "#9E9E9E") for l in lc.index]
wedges, texts, autotexts = ax1.pie(
    lc.values, labels=lc.index, autopct="%1.1f%%",
    colors=colors_pie, startangle=90, pctdistance=0.75,
    wedgeprops=dict(edgecolor="white", linewidth=2),
)
for at in autotexts:
    at.set_fontsize(10)
    at.set_fontweight("bold")
ax1.set_title(f"통화 유형 분포 (총 {len(df):,}건)", fontweight="bold", pad=10)

ax2 = axes[0, 1]
monthly = df.groupby("month").agg(
    총건수=("label", "count"),
    상담수=("label", lambda x: (x == "상담").sum()),
).reset_index()
monthly["상담률"] = monthly["상담수"] / monthly["총건수"] * 100
x = range(len(monthly))
ax2.bar(x, monthly["총건수"], color="#BBDEFB", label="전체", width=0.7)
ax2.bar(x, monthly["상담수"], color="#2196F3", label="상담", width=0.7)
ax2t = ax2.twinx()
ax2t.plot(x, monthly["상담률"], "ro-", linewidth=2, markersize=6, label="상담률(%)")
ax2t.set_ylim(0, 115)
ax2t.set_ylabel("상담률 (%)", color="red", fontsize=9)
ax2t.tick_params(axis="y", labelcolor="red")
ax2.set_xticks(list(x))
ax2.set_xticklabels([m[2:] for m in monthly["month"]], rotation=45, fontsize=8)
ax2.set_title("월별 통화 건수 및 상담률 추세", fontweight="bold")
ax2.set_ylabel("건수")
ax2.legend(loc="upper left", fontsize=8)
ax2t.legend(loc="upper right", fontsize=8)
for xi, rate in zip(x, monthly["상담률"]):
    ax2t.text(xi, rate + 3, f"{rate:.0f}%", ha="center", fontsize=7, color="red")

ax3 = axes[1, 0]
labels_order = ["상담", "비상담", "불명확"]
data_box = [df[df["label"] == l]["duration_sec"].dropna().clip(0, 600).values for l in labels_order]
bp = ax3.boxplot(data_box, labels=labels_order, patch_artist=True, medianprops=dict(color="black", linewidth=2))
box_colors = ["#2196F3", "#FF9800", "#9E9E9E"]
for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax3.set_title("통화 유형별 통화시간 분포 (초, 600초 상한)", fontweight="bold")
ax3.set_ylabel("통화시간 (초)")
for i, data in enumerate(data_box, 1):
    if len(data) > 0:
        ax3.text(i, np.mean(data) + 10, f"평균\n{np.mean(data):.0f}초", ha="center", fontsize=8, color="darkblue")

ax4 = axes[1, 1]
qa_dist = consult_df["qa_count"].value_counts().sort_index()
bars4 = ax4.bar(qa_dist.index, qa_dist.values, color="#2196F3", alpha=0.8, edgecolor="white")
ax4.set_title(f"상담 건당 QA 쌍 수 분포 (평균 {consult_df.qa_count.mean():.1f}개)", fontweight="bold")
ax4.set_xlabel("QA 쌍 수")
ax4.set_ylabel("상담 건수")
ax4.set_xticks(qa_dist.index)
for bar in bars4:
    h = bar.get_height()
    if h > 0:
        ax4.text(bar.get_x() + bar.get_width() / 2, h + 1, str(int(h)), ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/01_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {OUTPUT_DIR}/01_overview.png")


## 2. 상담 유형 자동 분류

In [ ]:
CATEGORY_KEYWORDS = {
    "인쇄/디자인/제작": ["인쇄", "제작", "디자인", "로고", "이미지", "파일", "시안", "일러스트", "실크", "색상", "도안", "인쇄물", "프린트", "ai파일", "pdf"],
    "주문/발주/견적": ["주문", "발주", "견적", "오더", "발주서", "주문서", "진행", "개 주문", "몇 개"],
    "배송/출고/납품": ["배송", "출고", "택배", "도착", "납품", "퀵", "당일", "출고일", "발송", "배달"],
    "가격/단가/수량": ["가격", "단가", "수량", "얼마", "가격표", "단가표", "개당", "비용", "몇개"],
    "샘플 요청": ["샘플", "견본", "시제품", "샘플비", "샘플 제작"],
    "결제/세금계산서": ["결제", "세금계산서", "영수증", "계산서", "입금", "카드", "계좌", "세금"],
    "상품 확인/재고": ["재고", "상품코드", "품절", "코드", "취급", "상품 있", "있나요", "있습니까"],
    "취소/환불/교환": ["취소", "환불", "반품", "교환"],
    "사이트/계정": ["로그인", "회원", "아이디", "비밀번호", "사이트", "홈페이지", "가입"],
}


def classify_row(row):
    text = str(row.get("transcript", "") or "") + " " + str(row.get("summary", "") or "")
    qa_pairs = row.get("qa_pairs", [])
    if isinstance(qa_pairs, list):
        for qa in qa_pairs:
            if isinstance(qa, dict):
                text += " " + (qa.get("question", "") or "") + " " + (qa.get("answer", "") or "")
    scores = {}
    for cat, kws in CATEGORY_KEYWORDS.items():
        s = sum(text.count(kw) for kw in kws)
        if s > 0:
            scores[cat] = s
    return max(scores, key=scores.get) if scores else "기타"


consult_df["category"] = consult_df.apply(classify_row, axis=1)

cat_counts = consult_df["category"].value_counts()
cat_list = cat_counts.index.tolist()
total_c = len(consult_df)

print("[ 상담 유형 분류 결과 ]")
print("=" * 55)
for cat, cnt in cat_counts.items():
    bar = "█" * int(cnt / total_c * 30)
    print(f"  {cat:<15}: {cnt:>4}건 ({cnt/total_c*100:5.1f}%)  {bar}")
print(f"  {'합계':<15}: {total_c:>4}건")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("상담 유형별 분포 분석", fontsize=15, fontweight="bold")

cat_vals = cat_counts.values.tolist()
colors_cat = CAT_COLORS[: len(cat_list)]

ax1 = axes[0]
bars = ax1.barh(cat_list[::-1], cat_vals[::-1], color=colors_cat[::-1], alpha=0.85, edgecolor="white")
for bar, val in zip(bars, cat_vals[::-1]):
    ax1.text(bar.get_width() + 2, bar.get_y() + bar.get_height() / 2, f"{val}건 ({val/total_c*100:.1f}%)", va="center", fontsize=9)
ax1.set_xlim(0, max(cat_vals) * 1.3)
ax1.set_title("상담 유형별 건수", fontweight="bold")
ax1.set_xlabel("건수")
ax1.tick_params(axis="y", labelsize=10)

ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    cat_vals, labels=None, autopct="%1.1f%%", colors=colors_cat, startangle=140,
    pctdistance=0.75, wedgeprops=dict(edgecolor="white", linewidth=2),
)
for at in autotexts:
    at.set_fontsize(8)
patches = [mpatches.Patch(color=colors_cat[i], label=cat_list[i]) for i in range(len(cat_list))]
ax2.legend(handles=patches, loc="lower right", fontsize=8, title="유형", title_fontsize=9)
ax2.set_title("상담 유형 구성 비율", fontweight="bold")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/02_category_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {OUTPUT_DIR}/02_category_distribution.png")


## 3. 월별 트렌드 및 유형별 변화

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 12))

ax1 = axes[0]
monthly_cat = consult_df.groupby(["month", "category"]).size().unstack(fill_value=0)
monthly_cat = monthly_cat.reindex(columns=cat_list, fill_value=0)

bottom = np.zeros(len(monthly_cat))
for i, cat in enumerate(cat_list):
    if cat in monthly_cat.columns:
        vals = monthly_cat[cat].values
        ax1.bar(range(len(monthly_cat)), vals, bottom=bottom, label=cat,
                color=CAT_COLORS[i % len(CAT_COLORS)], alpha=0.85, edgecolor="white", linewidth=0.5)
        bottom += vals

ax1.set_xticks(range(len(monthly_cat)))
ax1.set_xticklabels([m[2:] for m in monthly_cat.index], rotation=45, fontsize=9)
ax1.set_title("월별 상담 유형별 건수 (누적)", fontweight="bold", fontsize=13)
ax1.set_ylabel("건수")
ax1.legend(loc="upper left", fontsize=8, ncol=3)

ax2 = axes[1]
heatmap_data = monthly_cat.T
heatmap_pct = heatmap_data.div(heatmap_data.sum(axis=0), axis=1) * 100
heatmap_pct.columns = [m[2:] for m in heatmap_pct.columns]

sns.heatmap(
    heatmap_pct, ax=ax2, annot=True, fmt=".0f", cmap="YlOrRd",
    linewidths=0.5, linecolor="white", cbar_kws={"label": "비율(%)"}, annot_kws={"size": 8},
)
ax2.set_title("월별 상담 유형 비율 히트맵 (%)", fontweight="bold", fontsize=13)
ax2.set_xlabel("연월")
ax2.set_ylabel("상담 유형")
ax2.tick_params(axis="y", labelsize=9)
ax2.tick_params(axis="x", rotation=45, labelsize=8)

plt.tight_layout(pad=3)
plt.savefig(f"{OUTPUT_DIR}/03_monthly_trend.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {OUTPUT_DIR}/03_monthly_trend.png")


## 4. 통화 시간 심층 분석

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 6))
fig.suptitle("통화 시간 심층 분석", fontsize=14, fontweight="bold")

ax1 = axes[0]
durations = consult_df["duration_sec"].clip(0, 700)
ax1.hist(durations, bins=30, color="#2196F3", alpha=0.8, edgecolor="white")
ax1.axvline(durations.mean(), color="red", linestyle="--", linewidth=2, label=f"평균 {durations.mean():.0f}초")
ax1.axvline(durations.median(), color="orange", linestyle="--", linewidth=2, label=f"중앙값 {durations.median():.0f}초")
ax1.set_title("상담 통화시간 분포", fontweight="bold")
ax1.set_xlabel("통화시간 (초)")
ax1.set_ylabel("건수")
ax1.legend(fontsize=9)

ax2 = axes[1]
bins = [0, 30, 60, 120, 300, 600, float("inf")]
labels_b = ["~30초", "30~60초", "1~2분", "2~5분", "5~10분", "10분+"]
consult_df["dur_band"] = pd.cut(consult_df["duration_sec"], bins=bins, labels=labels_b)
band_counts = consult_df["dur_band"].value_counts().reindex(labels_b, fill_value=0)
colors_band = ["#FFCDD2", "#F8BBD9", "#CE93D8", "#90CAF9", "#A5D6A7", "#FFCC80"]
bars2 = ax2.bar(labels_b, band_counts.values, color=colors_band, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars2, band_counts.values):
    if val > 0:
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, f"{val}건\n({val/total_c*100:.0f}%)", ha="center", fontsize=8)
ax2.set_title("통화 시간 구간별 상담 건수", fontweight="bold")
ax2.set_xlabel("통화시간 구간")
ax2.set_ylabel("건수")
ax2.tick_params(axis="x", labelsize=9)

ax3 = axes[2]
scatter_data = consult_df[consult_df["qa_count"] <= 8].copy()
ax3.scatter(scatter_data["duration_sec"], scatter_data["qa_count"], alpha=0.4, c="#2196F3", s=30, edgecolors="none")
z = np.polyfit(scatter_data["duration_sec"], scatter_data["qa_count"], 1)
p = np.poly1d(z)
xline = np.linspace(0, 700, 100)
ax3.plot(xline, p(xline), "r--", linewidth=2, label="추세선")
corr = scatter_data["duration_sec"].corr(scatter_data["qa_count"])
ax3.set_title(f"통화시간 vs QA 수 (상관계수 r={corr:.2f})", fontweight="bold")
ax3.set_xlabel("통화시간 (초)")
ax3.set_ylabel("QA 수")
ax3.set_yticks(range(0, 9))
ax3.legend(fontsize=9)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/04_call_duration.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {OUTPUT_DIR}/04_call_duration.png")

print("\n[ 통화시간 구간별 분류 ]")
print(f"  30초 이하:  {band_counts['~30초']}건 → 챗봇 완전 처리 가능")
print(f"  1분 이하:   {band_counts['30~60초']}건 → 챗봇 처리 가능")
print(f"  2분 이하:   {band_counts['1~2분']}건 → 챗봇 + 확인 필요")
print(f"  5분 이하:   {band_counts['2~5분']}건 → 상담원 연결 권고")
print(f"  5분 초과:   {band_counts['5~10분']+band_counts['10분+']}건 → 상담원 연결 필수")


## 5. 키워드 분석

In [ ]:
KEYWORDS = [
    "주문", "발주", "견적", "오더", "납품",
    "배송", "택배", "출고", "도착", "발송", "퀵",
    "취소", "환불", "반품", "교환",
    "결제", "세금계산서", "영수증", "입금", "계좌",
    "샘플", "시제품",
    "상품", "제품", "가격", "단가", "수량", "재고",
    "로그인", "회원", "아이디", "비밀번호",
    "파일", "디자인", "인쇄", "제작", "로고", "이미지", "시안", "일러스트",
    "볼펜", "가방", "머그", "에코백", "마스크", "노트", "달력", "우산",
    "색상", "사이즈", "수정", "확인", "문의",
]

transcripts_all = " ".join(consult_df["transcript"].dropna().astype(str))
kw_freq = {kw: transcripts_all.count(kw) for kw in KEYWORDS if transcripts_all.count(kw) > 0}
kw_sorted = sorted(kw_freq.items(), key=lambda x: -x[1])

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle("상담 전사문 키워드 빈도 분석", fontsize=14, fontweight="bold")


def kw_color(kw):
    if kw in ["인쇄", "제작", "디자인", "로고", "이미지", "파일", "시안", "일러스트", "색상"]:
        return "#E91E63"
    if kw in ["주문", "발주", "견적", "오더", "납품"]:
        return "#2196F3"
    if kw in ["배송", "택배", "출고", "도착", "발송", "퀵"]:
        return "#4CAF50"
    if kw in ["가격", "단가", "수량", "재고"]:
        return "#FF9800"
    if kw in ["결제", "세금계산서", "영수증", "입금", "계좌"]:
        return "#00BCD4"
    if kw in ["취소", "환불", "반품"]:
        return "#795548"
    if kw in ["로그인", "회원", "아이디", "비밀번호"]:
        return "#9E9E9E"
    if kw in ["볼펜", "가방", "머그", "에코백", "마스크", "노트", "달력", "우산"]:
        return "#9C27B0"
    return "#607D8B"


ax1 = axes[0]
top30 = kw_sorted[:30]
kw_names = [k for k, v in top30]
kw_values = [v for k, v in top30]
colors_kw = [kw_color(k) for k in kw_names]
bars1 = ax1.barh(kw_names[::-1], kw_values[::-1], color=colors_kw[::-1], alpha=0.85, edgecolor="white")
for bar, val in zip(bars1, kw_values[::-1]):
    ax1.text(bar.get_width() + 3, bar.get_y() + bar.get_height() / 2, str(val), va="center", fontsize=8)
ax1.set_title("상위 30개 키워드 빈도", fontweight="bold")
ax1.set_xlabel("출현 횟수")
ax1.tick_params(axis="y", labelsize=9)
legend_items = [
    mpatches.Patch(color="#E91E63", label="인쇄/디자인/제작"),
    mpatches.Patch(color="#2196F3", label="주문/발주/견적"),
    mpatches.Patch(color="#4CAF50", label="배송/출고"),
    mpatches.Patch(color="#FF9800", label="가격/단가/수량"),
    mpatches.Patch(color="#00BCD4", label="결제/세금계산서"),
    mpatches.Patch(color="#9C27B0", label="상품 카테고리"),
]
ax1.legend(handles=legend_items, loc="lower right", fontsize=8)

ax2 = axes[1]
groups = {
    "인쇄/제작/디자인": ["인쇄", "제작", "디자인", "로고", "이미지", "파일", "시안", "일러스트"],
    "주문/발주/견적": ["주문", "발주", "견적", "납품", "오더"],
    "배송/출고": ["배송", "택배", "출고", "도착", "발송", "퀵"],
    "가격/단가": ["가격", "단가", "수량", "재고"],
    "결제/증빙": ["결제", "세금계산서", "영수증", "입금", "계좌"],
    "취소/환불": ["취소", "환불", "반품", "교환"],
    "샘플": ["샘플", "시제품"],
    "계정/사이트": ["로그인", "회원", "아이디", "비밀번호"],
}
group_totals = {g: sum(kw_freq.get(k, 0) for k in kws) for g, kws in groups.items()}
group_sorted = sorted(group_totals.items(), key=lambda x: -x[1])
g_names = [g for g, v in group_sorted]
g_values = [v for g, v in group_sorted]
g_colors = ["#E91E63", "#2196F3", "#4CAF50", "#FF9800", "#00BCD4", "#795548", "#9C27B0", "#9E9E9E"]
bars2 = ax2.bar(g_names, g_values, color=g_colors[: len(g_names)], alpha=0.85, edgecolor="white")
for bar, val in zip(bars2, g_values):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5, str(val), ha="center", fontsize=9, fontweight="bold")
ax2.set_title("상담 주제 그룹별 키워드 총 빈도", fontweight="bold")
ax2.set_ylabel("키워드 출현 합계")
ax2.tick_params(axis="x", rotation=35, labelsize=9)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/05_keyword_frequency.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {OUTPUT_DIR}/05_keyword_frequency.png")


## 6. 상담 유형별 복잡도 비교

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("상담 유형별 복잡도 비교", fontsize=14, fontweight="bold")

cat_stats2 = consult_df.groupby("category").agg(
    건수=("duration_sec", "count"),
    평균통화시간=("duration_sec", "mean"),
    평균QA수=("qa_count", "mean"),
).reset_index().sort_values("건수", ascending=False)

cats = cat_stats2["category"].tolist()
colors_cs = CAT_COLORS[: len(cats)]

ax1 = axes[0]
bars1 = ax1.barh(cats[::-1], cat_stats2["평균통화시간"].tolist()[::-1], color=colors_cs[::-1], alpha=0.85, edgecolor="white")
for bar, val in zip(bars1, cat_stats2["평균통화시간"].tolist()[::-1]):
    ax1.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2, f"{val:.0f}초", va="center", fontsize=9)
ax1.axvline(consult_df["duration_sec"].mean(), color="red", linestyle="--", linewidth=1.5,
            label=f"전체 평균 {consult_df['duration_sec'].mean():.0f}초")
ax1.set_title("유형별 평균 통화시간", fontweight="bold")
ax1.set_xlabel("초")
ax1.legend(fontsize=9)
ax1.tick_params(axis="y", labelsize=9)

ax2 = axes[1]
x_vals = cat_stats2["평균통화시간"].values
y_vals = cat_stats2["평균QA수"].values
sizes = cat_stats2["건수"].values
ax2.scatter(x_vals, y_vals, s=sizes * 3, c=colors_cs, alpha=0.7, edgecolors="white", linewidth=2)
for i, (x, y, cat, n) in enumerate(zip(x_vals, y_vals, cats, sizes)):
    ax2.annotate(f"{cat}\n({n}건)", (x, y), textcoords="offset points", xytext=(8, 4), fontsize=7.5, ha="left")
ax2.set_title("유형별 평균 통화시간 vs QA 수\n(버블 크기 = 건수)", fontweight="bold")
ax2.set_xlabel("평균 통화시간 (초)")
ax2.set_ylabel("평균 QA 수")
ax2.axhline(consult_df["qa_count"].mean(), color="gray", linestyle=":", linewidth=1)
ax2.axvline(consult_df["duration_sec"].mean(), color="gray", linestyle=":", linewidth=1)
ax2.text(0.02, 0.97, "↑ QA 많음, 짧음\n(챗봇 최적)", transform=ax2.transAxes, fontsize=8, va="top", color="green", alpha=0.7)
ax2.text(0.97, 0.97, "↑ QA 많음, 길음\n(상담원 필요)", transform=ax2.transAxes, fontsize=8, va="top", ha="right", color="red", alpha=0.7)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/06_category_complexity.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {OUTPUT_DIR}/06_category_complexity.png")


## 7. QA 패턴 분석

In [ ]:
if len(qa_df) == 0:
    print("QA 데이터 없음")
else:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle("QA 패턴 분석", fontsize=14, fontweight="bold")

    ax1 = axes[0, 0]
    ax1.hist(qa_df["q_len"].clip(0, 200), bins=30, color="#2196F3", alpha=0.7, edgecolor="white", label="질문")
    ax1.hist(qa_df["a_len"].clip(0, 200), bins=30, color="#4CAF50", alpha=0.5, edgecolor="white", label="답변")
    ax1.axvline(qa_df["q_len"].mean(), color="#1565C0", linestyle="--", linewidth=2, label=f"질문 평균 {qa_df['q_len'].mean():.0f}자")
    ax1.axvline(qa_df["a_len"].mean(), color="#2E7D32", linestyle="--", linewidth=2, label=f"답변 평균 {qa_df['a_len'].mean():.0f}자")
    ax1.set_title("QA 질문/답변 길이 분포", fontweight="bold")
    ax1.set_xlabel("글자 수")
    ax1.set_ylabel("빈도")
    ax1.legend(fontsize=8)

    ax2 = axes[0, 1]
    q_types = {
        "가능한가요/되나요": ["가능", "되나요", "됩니까", "돼요"],
        "언제 (배송/출고)": ["언제", "며칠", "몇 일"],
        "얼마 (가격)": ["얼마", "가격이", "단가가", "비용이"],
        "어떻게 (방법)": ["어떻게", "어떤 방법"],
        "있나요 (재고/상품)": ["있나요", "있습니까", "취급"],
        "변경/수정": ["변경", "수정", "바꿔"],
        "확인 요청": ["확인", "맞나요", "맞죠"],
        "문의/요청": ["문의", "요청", "부탁"],
    }
    q_type_counts = {}
    for qtype, kws in q_types.items():
        cnt = sum(qa_df["question"].str.contains(kw, na=False).sum() for kw in kws)
        if cnt > 0:
            q_type_counts[qtype] = cnt
    q_sorted = sorted(q_type_counts.items(), key=lambda x: -x[1])
    q_names = [q for q, v in q_sorted]
    q_values = [v for q, v in q_sorted]
    bars2 = ax2.bar(q_names, q_values, color=CAT_COLORS[: len(q_names)], alpha=0.85, edgecolor="white")
    for bar, val in zip(bars2, q_values):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, str(val), ha="center", fontsize=9)
    ax2.set_title("고객 질문 유형 분류", fontweight="bold")
    ax2.set_ylabel("QA 수")
    ax2.tick_params(axis="x", rotation=35, labelsize=8)

    ax3 = axes[1, 0]
    monthly_qa = qa_df.groupby("month").size().reset_index(name="qa수")
    ax3.bar(range(len(monthly_qa)), monthly_qa["qa수"], color="#2196F3", alpha=0.7, edgecolor="white")
    ax3.set_xticks(range(len(monthly_qa)))
    ax3.set_xticklabels([m[2:] for m in monthly_qa["month"]], rotation=45, fontsize=8)
    ax3.set_title("월별 QA 생성 건수", fontweight="bold")
    ax3.set_ylabel("QA 수")

    ax4 = axes[1, 1]
    a_bins = [0, 30, 60, 100, 150, float("inf")]
    a_labels = ["~30자", "30~60자", "60~100자", "100~150자", "150자+"]
    qa_df["a_band"] = pd.cut(qa_df["a_len"], bins=a_bins, labels=a_labels)
    a_dist = qa_df["a_band"].value_counts().reindex(a_labels, fill_value=0)
    colors_a = ["#BBDEFB", "#90CAF9", "#64B5F6", "#42A5F5", "#1E88E5"]
    bars4 = ax4.bar(a_labels, a_dist.values, color=colors_a, edgecolor="white")
    for bar, val in zip(bars4, a_dist.values):
        if val > 0:
            ax4.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, f"{val}개\n({val/len(qa_df)*100:.0f}%)", ha="center", fontsize=8)
    ax4.set_title("챗봇 답변 길이 분포", fontweight="bold")
    ax4.set_xlabel("답변 글자 수")
    ax4.set_ylabel("QA 수")

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/07_qa_patterns.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"저장: {OUTPUT_DIR}/07_qa_patterns.png")


## 8. 비상담 패턴 분석

In [ ]:
non_df = df[df["label"] == "비상담"].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("비상담 / 오인입 전화 분석", fontsize=14, fontweight="bold")

ax1 = axes[0]
oinin_keywords = ["병원", "올림픽", "잘못", "올리픽", "폐업", "아닌가요"]
ad_keywords = ["광고", "대행", "네이버", "마케팅"]
oinin_cnt = int(non_df["transcript"].fillna("").str.contains("|".join(oinin_keywords)).sum())
ad_cnt = int(non_df["transcript"].fillna("").str.contains("|".join(ad_keywords)).sum())
other_cnt = max(0, len(non_df) - oinin_cnt - ad_cnt)

categories_nc = ["병원 오인입", "광고/영업", "기타 비상담"]
values_nc = [oinin_cnt, ad_cnt, other_cnt]
colors_nc = ["#F44336", "#FF9800", "#9E9E9E"]
values_nc_pos = [v for v in values_nc if v > 0]
cats_nc_pos = [c for c, v in zip(categories_nc, values_nc) if v > 0]
colors_nc_pos = [c for c, v in zip(colors_nc, values_nc) if v > 0]

if values_nc_pos:
    wedges, texts, autotexts = ax1.pie(
        values_nc_pos, labels=cats_nc_pos, autopct="%1.1f%%", colors=colors_nc_pos, startangle=90,
        wedgeprops=dict(edgecolor="white", linewidth=2),
    )
ax1.set_title(f"비상담 유형 분류 (총 {len(non_df)}건)", fontweight="bold")

ax2 = axes[1]
monthly_nc = df.groupby("month").agg(
    상담=("label", lambda x: (x == "상담").sum()),
    비상담=("label", lambda x: (x == "비상담").sum()),
    불명확=("label", lambda x: (x == "불명확").sum()),
).reset_index()
x_nc = range(len(monthly_nc))
ax2.stackplot(x_nc, monthly_nc["비상담"], monthly_nc["불명확"], labels=["비상담", "불명확"],
              colors=["#FF9800", "#9E9E9E"], alpha=0.8)
ax2.set_xticks(list(x_nc))
ax2.set_xticklabels([m[2:] for m in monthly_nc["month"]], rotation=45, fontsize=8)
ax2.set_title("월별 비상담/불명확 추세", fontweight="bold")
ax2.set_ylabel("건수")
ax2.legend(loc="upper left", fontsize=9)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/08_non_consultation_patterns.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {OUTPUT_DIR}/08_non_consultation_patterns.png")
print(f"\n병원 오인입 추정: {oinin_cnt}건 (비상담의 {oinin_cnt/max(len(non_df),1)*100:.1f}%)")


## 9. 상품 카테고리 분석

In [ ]:
PRODUCTS = {
    "볼펜/펜": ["볼펜", "펜", "형광펜", "사인펜"],
    "머그컵/텀블러": ["머그", "텀블러", "컵", "머그컵"],
    "가방/에코백": ["가방", "에코백", "파우치", "백팩", "토트"],
    "인쇄물": ["인쇄물", "브로셔", "팸플릿", "명함", "스티커", "현수막"],
    "의류": ["티셔츠", "점퍼", "후드", "조끼", "유니폼", "의류"],
    "우산": ["우산", "양산"],
    "노트/다이어리": ["노트", "다이어리", "수첩"],
    "달력": ["달력", "캘린더"],
    "마스크": ["마스크"],
    "보조배터리": ["보조배터리", "배터리", "충전기"],
    "수건/타올": ["수건", "타올", "타월"],
    "USB": ["usb", "USB"],
}

transcripts_consult = " ".join(consult_df["transcript"].dropna().astype(str)).lower()
product_counts = {}
for prod, kws in PRODUCTS.items():
    cnt = sum(transcripts_consult.count(kw.lower()) for kw in kws)
    if cnt > 0:
        product_counts[prod] = cnt

product_sorted = sorted(product_counts.items(), key=lambda x: -x[1])
prod_names = [p for p, v in product_sorted]
prod_values = [v for p, v in product_sorted]

fig, ax = plt.subplots(figsize=(12, 6))
colors_prod = plt.cm.Set3(np.linspace(0, 1, len(prod_names)))
bars = ax.bar(prod_names, prod_values, color=colors_prod, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, prod_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5, str(val), ha="center", fontsize=9, fontweight="bold")
ax.set_title("상담에서 언급된 상품 카테고리 빈도 (상담 전사문 기준)", fontweight="bold", fontsize=13)
ax.set_xlabel("상품 카테고리")
ax.set_ylabel("언급 횟수")
ax.tick_params(axis="x", rotation=30, labelsize=10)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/09_product_categories.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {OUTPUT_DIR}/09_product_categories.png")

print("\n[ 상품별 언급 횟수 ]")
for prod, cnt in product_sorted:
    bar = "█" * min(cnt // 5, 40)
    print(f"  {prod:<15}: {cnt:>4}회  {bar}")


## 10. 챗봇 시나리오 우선순위 매트릭스

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
ax.set_facecolor("#F8F9FA")

priority_data = []
for i, cat in enumerate(cat_list):
    cat_data = consult_df[consult_df["category"] == cat]
    if len(cat_data) == 0:
        continue
    priority_data.append({
        "category": cat,
        "건수": len(cat_data),
        "평균통화시간": cat_data["duration_sec"].mean(),
        "평균QA수": cat_data["qa_count"].mean(),
    })

pdata = pd.DataFrame(priority_data)
x = pdata["건수"].values
y_raw = pdata["평균통화시간"].values
y = 300 - y_raw
sizes = pdata["평균QA수"].values * 300
cats_p = pdata["category"].tolist()

priority_score = x / max(x) * 0.6 + (y / max(np.abs(y) + 1)) * 0.4
priority_score = (priority_score - priority_score.min()) / (priority_score.max() - priority_score.min() + 1e-9)

sc = ax.scatter(x, y, s=sizes, c=priority_score, cmap="RdYlGn", alpha=0.8, edgecolors="white", linewidth=2, vmin=0, vmax=1)

for i, (xi, yi, cat) in enumerate(zip(x, y, cats_p)):
    ax.annotate(cat, (xi, yi), textcoords="offset points", xytext=(12, 0), fontsize=9, fontweight="bold",
                arrowprops=dict(arrowstyle="->", color="gray", lw=0.8))

plt.colorbar(sc, ax=ax, label="챗봇 자동화 우선순위 (높을수록 우선)")
ax.axhline(np.mean(y), color="gray", linestyle="--", linewidth=1, alpha=0.5)
ax.axvline(np.mean(x), color="gray", linestyle="--", linewidth=1, alpha=0.5)

ax.text(0.02, 0.97, "HIGH PRIORITY\n(빈도 낮음, 단순)\n→ 챗봇 보조", transform=ax.transAxes, fontsize=8, va="top", color="green", alpha=0.8)
ax.text(0.97, 0.97, "TOP PRIORITY\n(빈도 높음, 단순)\n→ 챗봇 핵심", transform=ax.transAxes, fontsize=8, va="top", ha="right", color="darkgreen", alpha=0.8, fontweight="bold")
ax.text(0.02, 0.03, "MEDIUM\n(빈도 낮음, 복잡)\n→ 상담원 연결", transform=ax.transAxes, fontsize=8, va="bottom", color="navy", alpha=0.8)
ax.text(0.97, 0.03, "ESCALATE\n(빈도 높음, 복잡)\n→ 상담원 연결 필수", transform=ax.transAxes, fontsize=8, va="bottom", ha="right", color="red", alpha=0.8)

ax.set_xlabel("상담 건수 (빈도)", fontsize=11)
ax.set_ylabel("자동화 적합도 (↑ 통화짧음 = 단순 문의)", fontsize=11)
ax.set_title("챗봇 시나리오 우선순위 매트릭스\n(버블 크기 = 평균 QA 수)", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/10_chatbot_priority_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {OUTPUT_DIR}/10_chatbot_priority_matrix.png")


## 11. 시간대별 상담 패턴

In [ ]:
from datetime import datetime


def extract_hour(fname):
    try:
        parts = str(fname).split("_")
        if len(parts) >= 7:
            return int(parts[5])
    except Exception:
        pass
    return None


def extract_weekday(fname):
    try:
        parts = str(fname).split("_")
        if len(parts) >= 8:
            d = datetime.strptime("_".join(parts[2:8]), "%Y_%m_%d_%H_%M_%S")
            return d.weekday()
    except Exception:
        pass
    return None


consult_df["hour"] = consult_df["file_name"].apply(extract_hour)
consult_df["weekday"] = consult_df["file_name"].apply(extract_weekday)

hour_valid = consult_df[consult_df["hour"].notna()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("시간대별 상담 패턴", fontsize=14, fontweight="bold")

ax1 = axes[0]
if len(hour_valid) > 0:
    hour_counts = hour_valid["hour"].value_counts().sort_index()
    all_hours = list(range(8, 20))
    hour_full = pd.Series([hour_counts.get(h, 0) for h in all_hours], index=all_hours)
    colors_h = [
        "#FFCDD2" if h < 9 or h >= 18 else "#90CAF9" if 9 <= h < 12 else "#A5D6A7" if 12 <= h < 14 else "#42A5F5"
        for h in all_hours
    ]
    bars_h = ax1.bar([f"{h}시" for h in all_hours], hour_full.values, color=colors_h, edgecolor="white")
    for bar, val in zip(bars_h, hour_full.values):
        if val > 0:
            ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5, str(val), ha="center", fontsize=8)
    legend_h = [
        mpatches.Patch(color="#FFCDD2", label="저녁/이른아침"),
        mpatches.Patch(color="#90CAF9", label="오전 (9~12시)"),
        mpatches.Patch(color="#A5D6A7", label="점심 (12~14시)"),
        mpatches.Patch(color="#42A5F5", label="오후 (14~18시)"),
    ]
    ax1.legend(handles=legend_h, fontsize=8)
ax1.set_title("시간대별 상담 건수", fontweight="bold")
ax1.set_ylabel("건수")
ax1.tick_params(axis="x", rotation=45, labelsize=8)

ax2 = axes[1]
wd_valid = consult_df[consult_df["weekday"].notna()]
if len(wd_valid) > 0:
    wd_counts = wd_valid["weekday"].value_counts().sort_index()
    wd_names = ["월", "화", "수", "목", "금", "토", "일"]
    wd_full = pd.Series([wd_counts.get(i, 0) for i in range(7)], index=wd_names)
    colors_wd = ["#42A5F5"] * 5 + ["#FFCDD2", "#FFCDD2"]
    bars_wd = ax2.bar(wd_names, wd_full.values, color=colors_wd, edgecolor="white")
    for bar, val in zip(bars_wd, wd_full.values):
        if val > 0:
            ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5, str(val), ha="center", fontsize=9)
ax2.set_title("요일별 상담 건수", fontweight="bold")
ax2.set_ylabel("건수")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/11_time_of_day.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {OUTPUT_DIR}/11_time_of_day.png")


## 저장 확인

In [ ]:
import glob

saved = sorted(glob.glob(f"{OUTPUT_DIR}/*.png"))
print(f"저장된 분석 이미지 ({len(saved)}개):")
for f in saved:
    print(f"  {f}")
